# Analyse RMSE — Osiris vs Fine Tuning

Ce notebook produit des tableaux de **RMSE** (m³/m³) avec un filtre sur les **sondes** et une vue **RMSE par catégorie de sonde** :

1. **RMSE par champ de test** — ligne = champ (`OSIRIS_<site>`), colonnes = modèle × profondeur.
2. **RMSE par champ de validation** — ligne = `val_site`, colonnes = modèle × profondeur.
3. **RMSE par catégorie de sonde** (en bas) — ligne = catégorie, colonnes = modèle × profondeur.

### Règles de codage
- `model = lstm` → **Osiris** | `model = lstm_fine_tuned` → **Fine Tuning**
- `type = OSIRIS_<année>_<champ>_<sonde>`
- **Catégorie de sonde** :
  - noms chiffrés (2024/2026) → `Numérique`
  - nom commençant par `Canon` / `Robot` / `Sec` et **sans `-`** → regroupés par premier mot (`Canon_1`, `Canon_2` → `Canon`)
  - nom contenant **`-`** → catégorie à part entière (`Robot_-20`, `Robot_-30`, …)
- Les valeurs **élevées** (RMSE ≥ seuil quantile) sont mises en **gras**.

In [1]:
import numpy as np
import pandas as pd
from IPython.display import display, Markdown
import ipywidgets as widgets

# ── Configuration ─────────────────────────────────────────────
RESULTS_CSV = "/home/theodore/Téléchargements/results.csv"
MODEL_LABELS = {"lstm": "Osiris", "lstm_fine_tuned": "Fine Tuning"}
THRESHOLD_MODE = "quantile"      # ou "mean_std"
THRESHOLD_QUANTILE = 0.75
THRESHOLD_N_STD = 1.0
PRECISION = 3

In [2]:
# ── Chargement et préparation ─────────────────────────────────
df = pd.read_csv(RESULTS_CSV, low_memory=False)
print(f"{len(df):,} lignes chargées.")

# Valeur RMSE = horizons séparés par des virgules
def parse_values(s):
    if isinstance(s, str) and s != "":
        try:
            return [float(x) for x in s.split(",")]
        except ValueError:
            return []
    return []

df["_horizons"] = df["value"].apply(parse_values)
df["rmse_mean"] = df["_horizons"].apply(lambda v: float(np.mean(v)) if v else np.nan)

# Lignes RMSE OSIRIS valides uniquement
rmse = df[(df["metric_name"] == "rmse_osiris") & df["rmse_mean"].notna()].copy()

# ── Champ (site) : via val_site connus ─────────────────────────
site_ids = sorted(rmse["val_site"].unique())

def extract_champ(typ):
    suffix = typ.replace("OSIRIS_", "")
    for sid in sorted(site_ids, key=len, reverse=True):
        if suffix == sid or suffix.startswith(sid + "_"):
            return "OSIRIS_" + sid
    return typ

rmse["champ"] = rmse["type"].apply(extract_champ)

# ── Sonde et catégorie de sonde ────────────────────────────────
def extract_probe(typ):
    champ = extract_champ(typ)            # "OSIRIS_2025_Grandvillers"
    return typ[len(champ) + 1:]           # "Robot_-20_2"

def make_probe_cat(probe):
    if probe.strip("0123456789") == "":
        return "Numérique"
    parts = probe.split("_")
    while len(parts) > 1 and parts[-1].isdigit():   # retire les index dédoublonnés (Canon_1, Sec_2, Robot_-20_2)
        parts.pop()
    cleaned = "_".join(parts)
    if "-" in cleaned:                               # catégorie à part dès qu'il y a un "-"
        return cleaned
    return parts[0]                                  # sinon premier mot (Canon / Robot / Sec)

rmse["probe_name"] = rmse["type"].apply(extract_probe)
rmse["probe_cat"] = rmse["probe_name"].apply(make_probe_cat)

# ── Synthèse ───────────────────────────────────────────────────
depths = sorted(rmse["depth"].unique())
models = [m for m in ["lstm", "lstm_fine_tuned"] if m in rmse["model"].unique()]
print(f"Champs            : {rmse['champ'].nunique()}")
print(f"Profondeurs       : {[float(d) for d in depths]}")
print(f"Modèles           : {[MODEL_LABELS.get(m, m) for m in models]}")
print(f"Lignes RMSE       : {len(rmse):,}")
print("\nCatégories de sonde :")
for c, n in rmse["probe_cat"].value_counts().items():
    print(f"  {c:12s} : {n}")
print("\nDétail sondes 2025 =")
for p in sorted(rmse.groupby('probe_name').size().index):
    cat = make_probe_cat(p)
    mark = "  <-- catégorie à part (contient -)" if "-" in p else ""
    print(f"  {p:35s} -> {cat:12s}{mark}")

15,798 lignes chargées.
Champs            : 13
Profondeurs       : [0.1, 0.2, 0.3, 0.4, 0.5]
Modèles           : ['Osiris']
Lignes RMSE       : 2,592

Catégories de sonde :
  Numérique    : 1680
  Robot        : 276
  Canon        : 240
  Sec          : 180
  Robot_-20    : 96
  Robot_-15    : 60
  Robot_-30    : 60

Détail sondes 2025 =
  1003168                             -> Numérique   
  887587                              -> Numérique   
  887717                              -> Numérique   
  891481                              -> Numérique   
  891515                              -> Numérique   
  892571                              -> Numérique   
  893569                              -> Numérique   
  895719                              -> Numérique   
  896077                              -> Numérique   
  996025                              -> Numérique   
  996039                              -> Numérique   
  996128                              -> Numérique   
  996652    

In [3]:
# ── Helpers : pivot + style ────────────────────────────────────
def build_multi_pivot(data, index_col):
    """Pivot lignes = index_col, colonnes = (Modèle, Profondeur)."""
    grp = (data.groupby([index_col, "model", "depth"])["rmse_mean"]
           .mean().reset_index())
    pivot = grp.pivot_table(index=index_col, columns=["model", "depth"],
                            values="rmse_mean", aggfunc="first")
    pivot.columns = pd.MultiIndex.from_tuples(
        [(MODEL_LABELS.get(m, m), f"{float(d):g}m") for m, d in pivot.columns],
        names=["Modèle", "Profondeur"])
    return pivot.sort_index()

def highlight_high(row, thr):
    return ["font-weight: bold" if pd.notna(v) and v >= thr else "" for v in row]

def style_multi_pivot(pivot):
    vals = pivot.to_numpy(dtype=float)
    vals = vals[~np.isnan(vals)]
    if len(vals) == 0:
        return pivot.style, float("nan")
    if THRESHOLD_MODE == "mean_std":
        thr = float(np.mean(vals) + THRESHOLD_N_STD * np.std(vals))
    else:
        thr = float(np.quantile(vals, THRESHOLD_QUANTILE))
    styled = pivot.style.apply(lambda r: highlight_high(r, thr), axis=1)
    styled = styled.format("{:.3f}", na_rep="\u2013")
    return styled, thr

def show_pivot(title, data, index_col):
    display(Markdown(f"### {title}"))
    pivot = build_multi_pivot(data, index_col)
    styled, thr = style_multi_pivot(pivot)
    print(f"Seuil de mise en gras : RMSE >= {thr:.3f}" if thr == thr else "Aucune donnée.")
    display(styled)

## Filtres

- **Sondes** : catégories `Canon`, `Robot`, `Sec`, les catégories contenant `-` (`Robot_-20`, …), et `Numérique`.
- **Modèle** : Osiris et/ou Fine Tuning.
- **Profondeur** : toutes ou une seule.

Ajustez les filtres : tous les tableaux ci-dessous (dont la RMSE par catégorie de sonde, en bas) se mettent à jour.

In [4]:
cat_sel  = widgets.SelectMultiple(
    options=sorted(rmse["probe_cat"].unique()),
    value=tuple(sorted(rmse["probe_cat"].unique())),
    description="Sondes :", layout=widgets.Layout(width="100%", height="110px"))
model_sel = widgets.SelectMultiple(
    options=[MODEL_LABELS[m] for m in models],
    value=tuple(MODEL_LABELS[m] for m in models),
    description="Modèle :", layout=widgets.Layout(width="100%"))
depth_sel = widgets.Dropdown(
    options=["Tous"] + [f"{float(d):g}" for d in depths],
    value="Tous", description="Profondeur :", layout=widgets.Layout(width="100%"))

def render(cats, mods, depth):
    data = rmse[rmse["probe_cat"].isin(list(cats))]
    data = data[data["model"].isin([k for k, v in MODEL_LABELS.items() if v in mods])]
    if depth != "Tous":
        data = data[data["depth"] == float(depth)]
    if len(data) == 0:
        display(Markdown("Aucune ligne ne correspond aux filtres."))
        return
    show_pivot("1. RMSE par champ de test", data, "champ")
    show_pivot("2. RMSE par champ de validation", data, "val_site")
    show_pivot("3. RMSE par catégorie de sonde", data, "probe_cat")

ui  = widgets.VBox([cat_sel, model_sel, depth_sel])
out = widgets.interactive_output(render, {"cats": cat_sel, "mods": model_sel, "depth": depth_sel})
display(ui, out)

Output()